# Colab Runner: `resnet18_covidqu_syn`

Run this notebook after the improved DCGAN images have been generated and inspected. Pretraining and fine-tuning are split into separate cells so you can run them in chunks.


## 1. Setup Repository


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, csv, json

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    os.chdir(REPO_ROOT)
    subprocess.run(['git', 'pull'], check=True)
else:
    os.chdir('/content')
    subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir(REPO_ROOT)

print('REPO_ROOT:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


## 2. Install Dependencies


In [ ]:
requirements_path = REPO_ROOT / 'requirements.txt'
filtered_requirements_path = REPO_ROOT / 'requirements_colab_filtered.txt'
skip_prefixes = ('torch', 'torchvision', 'torchaudio', 'triton', 'nvidia-')
if requirements_path.exists():
    filtered_lines, skipped_lines = [], []
    for raw_line in requirements_path.read_text().splitlines():
        line = raw_line.strip()
        package_name = line.split('==')[0].split('>=')[0].split('<=')[0].split('~=')[0].lower()
        if not line or line.startswith('#'):
            filtered_lines.append(raw_line)
        elif package_name.startswith(skip_prefixes):
            skipped_lines.append(raw_line)
        else:
            filtered_lines.append(raw_line)
    filtered_requirements_path.write_text('\n'.join(filtered_lines) + '\n')
    print('Skipped runtime packages:', skipped_lines)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(filtered_requirements_path)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'scikit-learn', 'matplotlib', 'pandas', 'Pillow', 'pyyaml'], check=True)

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 3. Edit Paths and Runtime Switches


In [ ]:
EXPERIMENT_ID = 'resnet18_covidqu_syn'
CONFIG_PATH = Path('configs/experiments/resnet18/covidqu_syn.yaml')
OUTPUT_ROOT = Path('/content/drive/MyDrive/medcls_cvproject/results/experiments_rerun_dcgan128')
OUT = OUTPUT_ROOT / EXPERIMENT_ID

# Use the improved Stage 1 output. Change this only if you saved the new DCGAN images elsewhere.
SYNTHETIC_DIR = Path('/content/drive/MyDrive/medcls_cvproject/data/processed/synthetic_dcgan_stabilized')
SYNTHETIC_MANIFEST = REPO_ROOT / 'data/manifests/synthetic_dcgan.csv'

RUN_PRETRAIN = True
RUN_FINETUNE = True
PRETRAIN_EPOCHS = None   # None means use config, currently 70 for ResNet SimCLR.
FINETUNE_EPOCHS = None   # None means use config, currently 70.
NUM_WORKERS = 2

print('EXPERIMENT_ID:', EXPERIMENT_ID)
print('CONFIG_PATH:', CONFIG_PATH)
print('OUT:', OUT)
print('SYNTHETIC_DIR:', SYNTHETIC_DIR, SYNTHETIC_DIR.exists())


## 4. Link Synthetic Data and Create Manifest


In [ ]:
CLASSES = ['COVID', 'Lung_Opacity', 'Viral_Pneumonia', 'Normal']
CLASS_TO_LABEL = {'COVID': 0, 'Lung_Opacity': 1, 'Viral_Pneumonia': 2, 'Normal': 3}
IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def list_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXTS)

def class_image_dir(root, cls):
    croot = Path(root) / cls
    return croot / 'images' if (croot / 'images').exists() else croot

def replace_path(target: Path, source: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    if source.exists():
        os.symlink(source, target, target_is_directory=source.is_dir())
        print('Linked', target, '->', source)
    else:
        raise FileNotFoundError(f'Missing source: {source}')

def write_synthetic_manifest(synthetic_root: Path, manifest_path: Path):
    rows = []
    for cls in CLASSES:
        files = list_images(class_image_dir(synthetic_root, cls))
        print(cls, len(files))
        for path in files:
            rel = Path('data/processed/synthetic_dcgan') / cls / 'images' / path.name
            rows.append({
                'image_path': str(rel),
                'class_name': cls,
                'label': CLASS_TO_LABEL[cls],
                'source': 'stage1_synthesis',
                'generator': 'DCGAN',
            })
    if not rows:
        raise ValueError(f'No synthetic images found at {synthetic_root}')
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['image_path', 'class_name', 'label', 'source', 'generator'])
        writer.writeheader()
        writer.writerows(rows)
    print('Wrote synthetic manifest:', manifest_path, 'rows=', len(rows))

os.chdir(REPO_ROOT)
print('Working directory:', Path.cwd())
replace_path(REPO_ROOT / 'data/processed/synthetic_dcgan', SYNTHETIC_DIR)
write_synthetic_manifest(REPO_ROOT / 'data/processed/synthetic_dcgan', SYNTHETIC_MANIFEST)
!python scripts/check_experiment_inputs.py --synthetic-manifest data/manifests/synthetic_dcgan.csv


## 5. Pretrain ResNet18 with SimCLR on DCGAN Synthetic Images


In [ ]:
pretrain_cmd = [
    sys.executable, 'scripts/run_simclr_resnet.py',
    '--config', str(CONFIG_PATH),
    '--synthetic-manifest', 'data/manifests/synthetic_dcgan.csv',
    '--output-dir', str(OUT),
    '--resume-checkpoint', str(OUT / 'pretrain/checkpoints/last_simclr_checkpoint.pth'),
    '--num-workers', str(NUM_WORKERS),
]
if PRETRAIN_EPOCHS is not None:
    pretrain_cmd += ['--epochs', str(PRETRAIN_EPOCHS)]
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'

if RUN_PRETRAIN:
    subprocess.run(pretrain_cmd, check=True)
else:
    print('Skipping pretraining')
print('Expected checkpoint:', CKPT, CKPT.exists())


## 6. Fine-Tune and Evaluate on Fixed Real Train/Val/Test Split


In [ ]:
finetune_cmd = [
    sys.executable, 'scripts/run_classification_resnet.py',
    '--config', str(CONFIG_PATH),
    '--manifest-dir', 'data/manifests',
    '--output-dir', str(OUT),
    '--pretrained-checkpoint', str(OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'),
    '--num-workers', str(NUM_WORKERS),
]
if FINETUNE_EPOCHS is not None:
    finetune_cmd += ['--epochs', str(FINETUNE_EPOCHS)]
CKPT = OUT / 'pretrain/checkpoints/best_simclr_backbone.pth'

if RUN_FINETUNE:
    if not CKPT.exists():
        raise FileNotFoundError(f'Missing pretrain checkpoint: {CKPT}')
    subprocess.run(finetune_cmd, check=True)
else:
    print('Skipping fine-tuning')


## 7. Display Results


In [ ]:
metrics_path = OUT / 'metrics.json'
if metrics_path.exists():
    import pandas as pd, json
    display(pd.DataFrame([{**{'experiment_id': EXPERIMENT_ID}, **json.loads(metrics_path.read_text())}]))
else:
    print('metrics.json not found:', metrics_path)
!find "{OUT}" -maxdepth 4 -type f | sort
